In [1]:
import warnings
warnings.filterwarnings( 'ignore' )

In [3]:
import pandas as  pd
import numpy as np
import pickle
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from nltk.corpus import stopwords
import nltk
import unicodedata

In [ ]:
path = 'D:/GFP/'
dataoriginal = pd.read_stata(path) 

In [ ]:
data = dataoriginal

In [ ]:
path = 'D:/GFP/renamu_diccionario_clasificacion.xlsx'
varnames = pd.read_excel(path, engine='openpyxl')

**1. Exploración de la data**

In [ ]:
# Filtrar las variables que no deben usarse (usar == 'NO')
vars_to_exclude = varnames[varnames['usar'] == 'NO']['variable'].tolist()
data = data.drop(columns=vars_to_exclude)
data.info()

In [ ]:
import pandas as pd

# Asegúrate de que tipo_agregacion esté bien definido (rellenando NaN con 'ninguna')
varnames['tipo_agregacion'] = varnames['tipo_agregacion'].fillna('ninguna')

# Agregar todas las variables dentro de cada bloque según tipo de agregación
for bloque in varnames['bloque'].unique():
    # Filtrar las variables del bloque actual
    bloque_vars = varnames[varnames['bloque'] == bloque]
    
    # Sumar las variables de tipo "sumar" dentro de este bloque
    if 'sumar' in bloque_vars['tipo_agregacion'].values:
        # Filtrar solo las variables con tipo "sumar"
        vars_a_sumar = bloque_vars[bloque_vars['tipo_agregacion'] == 'sumar']['variable']
        
        # Verificar que las variables existan en 'data'
        vars_a_sumar = [var for var in vars_a_sumar if var in data.columns]
        
        # Asegúrate de que las variables son numéricas
        data[vars_a_sumar] = data[vars_a_sumar].apply(pd.to_numeric, errors='coerce')

        if vars_a_sumar:
            # Sumar todas las variables seleccionadas
            data[f'{bloque}_sum'] = data[vars_a_sumar].sum(axis=1, skipna=True)
            data.drop(columns=vars_a_sumar, inplace=True)  # Eliminar las variables sumadas
    
    # Sumar las operativas dentro del bloque
    if 'sumar_operativas' in bloque_vars['tipo_agregacion'].values:
        # Filtrar solo las variables con "sumar_operativas"
        vars_operativas = bloque_vars[bloque_vars['tipo_agregacion'] == 'sumar_operativas']['variable']
        
        # Verificar que las variables operativas existan en 'data'
        vars_operativas = [var for var in vars_operativas if var in data.columns]
        
        # Asegúrate de que las variables son numéricas
        data[vars_operativas] = data[vars_operativas].apply(pd.to_numeric, errors='coerce')
        
        if vars_operativas:
            # Sumar solo las operativas
            data[f'{bloque}_operativas_sum'] = data[vars_operativas].sum(axis=1, skipna=True)
            data.drop(columns=vars_operativas, inplace=True)  # Eliminar las variables operativas
    
    # Sumar las no operativas dentro del bloque
    if 'sumar_no_operativas' in bloque_vars['tipo_agregacion'].values:
        # Filtrar solo las variables con "sumar_no_operativas"
        vars_no_operativas = bloque_vars[bloque_vars['tipo_agregacion'] == 'sumar_no_operativas']['variable']
        
        # Verificar que las variables no operativas existan en 'data'
        vars_no_operativas = [var for var in vars_no_operativas if var in data.columns]
        
        # Asegúrate de que las variables son numéricas
        data[vars_no_operativas] = data[vars_no_operativas].apply(pd.to_numeric, errors='coerce')
        
        if vars_no_operativas:
            # Sumar solo las no operativas
            data[f'{bloque}_no_operativas_sum'] = data[vars_no_operativas].sum(axis=1, skipna=True)
            data.drop(columns=vars_no_operativas, inplace=True)  # Eliminar las variables no operativas

# Ver las primeras filas del dataframe con las sumas realizadas

In [ ]:
data.columns.tolist()

**5. Filtros pendientes**

In [ ]:
# 1. Filtrar Missings: columnas con más del 10% de valores faltantes
threshold = 0.1  # Umbral de valores faltantes permitido
missing_ratio = data.isna().mean()  # Calcula el porcentaje de valores faltantes por columna
data = data.loc[:, missing_ratio <= threshold]  # Mantiene solo las columnas con menos del umbral

In [ ]:
# Definir columnas geográficas para la imputación
columnas_geo = ['Departamento', 'Provincia', 'Distrito', 'year']
variables_num = data.select_dtypes(include=['number']).columns.tolist()
data[variables_num] = data.groupby(columnas_geo)[variables_num].transform(lambda x: x.fillna(x.mean()))

In [ ]:
# Paso 2: Imputar los NaN restantes con media global
data[variables_num] = data[variables_num].fillna(data[variables_num].mean())

In [ ]:
# 2. Filtro de Variabilidad: Columnas que solo tienen una categoria, esas se eliminan
variables_cat = data.select_dtypes(include=['category', 'object']).columns.tolist()
columnas_baja_variabilidad = [col for col in variables_cat if data[col].nunique(dropna=True) == 1]
data = data.drop(columns=columnas_baja_variabilidad)

In [ ]:
# Paso 3: Imputación por moda a nivel (departamento, provincia, distrito)
columnas_geo = ['Departamento', 'Provincia', 'Distrito', 'year']
# Seleccionar variables categóricas a imputar
variables_cat = data.select_dtypes(include=['object', 'category']).columns.tolist()
variables_cat = [col for col in variables_cat if col not in columnas_geo]  # excluir claves geográficas

# Imputar por moda dentro de cada grupo único (departamento, provincia, distrito)
for col in variables_cat:
    try:
        data[col] = data.groupby(columnas_geo)[col].transform(
            lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x
        )
    except Exception as e:
        print(f"No se pudo imputar la variable '{col}': {e}")

In [ ]:
# Paso 4: Imputación por moda a global (departamento, provincia, distrito)
for col in variables_cat:
    if data[col].isna().sum() > 0:
        try:
            moda_global = data[col].mode().iloc[0]
            data[col] = data[col].fillna(moda_global)
        except Exception as e:
            print(f"No se pudo imputar globalmente la variable '{col}': {e}")

In [ ]:
# Paso 5: Filtro de Variabilidad: Columnas que tienen 0.1 por ciento en comparación al total de casos
variables_cat = data.select_dtypes(include=['category', 'object']).columns.tolist()
# Umbral de cardinalidad
umbral_cardinalidad = 200
# Identificar columnas de alta cardinalidad (excepto 'distrito')
columnas_alta_cardinalidad = [
    col for col in variables_cat 
    if data[col].nunique(dropna=True) > umbral_cardinalidad and col != 'Distrito'
]
# Eliminar columnas seleccionadas
data = data.drop(columns=columnas_alta_cardinalidad, errors='ignore')

In [ ]:
# 5. En este paso se excluyen variables que son consecuencia de la var dependiente y por tanto van a afectar inevitablemente el modelo. Esto se definio en base a cuestiones previas, se corrio el modelo y se identificaron esos resultados. 
# Lista de variables a excluir por data leakage
variables_a_excluir = [
    'direc_muni',
    'facebook', 
    'web',     
]

data = data.drop(columns=variables_a_excluir, errors='ignore')

In [ ]:
#data = data.drop(columns=["tipo_obra_nivel1", "tipo_obra_nivel2", "tipo_obra_nivel3"])

**6. Conversión a dicotómicas**

In [ ]:
variables_cat = data.select_dtypes(include=['object', 'category']).columns.tolist()
variables_dic = [col for col in variables_cat if data[col].nunique(dropna=True) == 2]

for col in variables_dic:
    try:
        # pd.factorize devuelve [0,1,...] y -1 si es NaN
        data[col], _ = pd.factorize(data[col])
        data[col] = data[col].replace(-1, np.nan)  # volver NaN si los había
    except Exception as e:
        print(f"No se pudo convertir la variable '{col}': {e}")

In [ ]:
# Variables categóricas
variables_cat = data.select_dtypes(include=['object', 'category']).columns.tolist()
# Variables a excluir del one-hot encoding
excluir = ['ccdd',
 'ccpp',
 'ccdi',
 'Departamento',
 'Provincia',
 'Distrito',
 'year',]
# Variables politómicas (más de 2 categorías), pero sin las excluidas
variables_pol = [col for col in variables_cat 
                 if data[col].nunique(dropna=True) > 2 and col not in excluir]
# Convertir a dummies solo las que quieres
data = pd.get_dummies(data, columns=variables_pol, drop_first=True)
# Asegurar que columnas booleanas queden como 0/1
cols_booleanas = data.select_dtypes(include=["bool"]).columns
data[cols_booleanas] = data[cols_booleanas].astype(int)

In [ ]:
data.columns.tolist()

In [ ]:
# Convertir montos a log

In [ ]:
path = 'D:/GFP/data_contrata_concatenacion.xlsx'
data2 = pd.read_excel(path, engine='openpyxl')

In [ ]:
data_filtrada = data2

In [ ]:
# renombrar
rename_map = {
    'provincia_norm': 'Provincia',
    'departamento':   'Departamento',
    'distrito':       'Distrito',
    'anio_inicio_obra':'year',   # <- solo esta cambia a Year
    # 'brecha_existente' se queda igual
}
data_filtrada = data_filtrada.rename(columns=rename_map)

print(data_filtrada.head())

In [ ]:
# --- Función para normalizar textos ---
def normalize_text(s):
    if pd.isna(s): 
        return None
    s = str(s).strip().upper()
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    return None if s in ['NAN','NULL','NONE',''] else s

# --- Normalizar claves en ambas bases ---
for col in ['Provincia','Distrito','Departamento']:
    data[col] = data[col].map(normalize_text)
    data_filtrada[col] = data_filtrada[col].map(normalize_text)

In [ ]:
data_filtrada_2018_2020 = data_filtrada[
    (data_filtrada["year"] >= 2018) & (data_filtrada["year"] <= 2020)
].copy()

In [ ]:
data_filtrada_2018_2020.info()

In [ ]:
merged = pd.merge(
    data_filtrada_2018_2020, data,
    on=['Departamento','Provincia','Distrito','year'],
    how='left', validate='m:m'
)

In [ ]:
merged.info()

In [ ]:
# 1. Ver resumen general
print(merged.info())
# 2. Contar valores nulos por columna
print("\nValores nulos por columna:")
print(merged.isnull().sum())
# 3. Identificar columnas completamente vacías (todas NaN)
cols_vacias = merged.columns[merged.isnull().all()].tolist()
print("\nColumnas completamente vacías:", cols_vacias)
# 4. Identificar columnas con muy pocos datos (ej. menos del 5% no nulos)
umbral = 0.05
cols_pocas_obs = merged.columns[(merged.notnull().mean() < umbral)].tolist()
print("\nColumnas con menos del 5% de observaciones válidas:", cols_pocas_obs)

In [ ]:
merged.info()

In [ ]:
###. PASO OPCIONNAL!!!

In [ ]:
# Variables que quieres excluir del filtro, pero que sí deben estar en el dataframe final
excluir = ['Provincia',
 'Distrito',
 'Departamento',
 'year',
 'ccdd',
 'ccpp',
 'ccdi',]

# Paso 1: Matriz de correlación (solo numéricas)
corr_matrix = merged.corr(numeric_only=True)

# Paso 2: Correlación de cada variable con la dependiente
cor_with_y = corr_matrix['brecha_existente'].drop('brecha_existente', errors='ignore')

# Paso 3: Variables independientes excluyendo las de 'excluir'
indep_vars = [v for v in cor_with_y.index if v not in excluir]
corr_indep = corr_matrix.loc[indep_vars, indep_vars]

# Paso 4: Eliminar variables muy correlacionadas (>= 0.85)
vars_to_remove = set()
threshold = 0.85
for i in range(len(indep_vars)):
    for j in range(i + 1, len(indep_vars)):
        var1, var2 = indep_vars[i], indep_vars[j]
        r = abs(corr_indep.loc[var1, var2])
        if r >= threshold:
            if abs(cor_with_y[var1]) >= abs(cor_with_y[var2]):
                vars_to_remove.add(var2)
            else:
                vars_to_remove.add(var1)

# Paso 5: Variables seleccionadas después del filtro
vars_selected = [v for v in indep_vars if v not in vars_to_remove]

# Paso 6: DataFrame final con seleccionadas + dependiente + excluidas
cols_finales = vars_selected + ['brecha_existente'] + [c for c in excluir if c in data.columns]
data_preseleccionada = merged[cols_finales].copy()


In [ ]:
#data.to_csv("full_data.csv", index=False, encoding="latin1")

In [ ]:
data_preseleccionada.info()

In [ ]:
data_preseleccionada.info()

In [ ]:
import pandas as pd
# Lista para guardar los detalles de exclusión
exclusion_log = []

# Repetimos la lógica del filtro, pero esta vez guardando la info de cada exclusión
for i in range(len(indep_vars)):
    for j in range(i + 1, len(indep_vars)):
        var1 = indep_vars[i]
        var2 = indep_vars[j]
        r = abs(corr_indep.loc[var1, var2])
        if r >= threshold:
            cor1 = abs(cor_with_y[var1])
            cor2 = abs(cor_with_y[var2])
            excluida = var2 if cor1 >= cor2 else var1
            conservada = var1 if excluida == var2 else var2
            exclusion_log.append({
                "var1": var1,
                "var2": var2,
                "correlacion_entre_ellas": r,
                "cor_var1_con_brecha": cor1,
                "cor_var2_con_brecha": cor2,
                "variable_conservada": conservada,
                "variable_excluida": excluida
            })

# Convertir a DataFrame
exclusion_df = pd.DataFrame(exclusion_log)
# Exportar a Excel
exclusion_df.to_excel("detalle_variables_excluidas.xlsx", index=False)

In [ ]:
data_preseleccionada.to_excel("6_data_full_renamu_clasificacion.xlsx", index=False, engine="openpyxl")